# 实验四参考答案：NLP 进阶与模型微调

本 notebook 提供「实验四」三个任务的完整参考实现：

- **任务1**：Transformer 自注意力机制实现（NumPy）
- **任务2**：BERT 情感分类微调（hotel.csv）
- **任务3（选做）**：LoRA 轻量微调对比

> 注意：参考答案仅供学习参考，鼓励同学先独立完成再对照。


## 任务1：Transformer 自注意力机制实现

**目标**：用 NumPy 手动实现单头自注意力（Scaled Dot-Product Attention）

**核心公式**：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

### 1.1 softmax 与自注意力实现


In [1]:
import numpy as np

def softmax(x):
    """对矩阵最后一个维度做 softmax（含数值稳定处理）"""
    # 减去最大值防止 exp 溢出（数值稳定技巧）
    x = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    """
    计算缩放点积注意力
    参数：
        Q: (seq_len, d_k) 查询矩阵
        K: (seq_len, d_k) 键矩阵
        V: (seq_len, d_v) 值矩阵
    返回：
        output: (seq_len, d_v) 注意力输出
        attention_weights: (seq_len, seq_len) 注意力权重
    """
    d_k = K.shape[-1]

    # Step 1: 计算 Q 和 K^T 的点积，得到相关性得分
    scores = Q @ K.T  # shape (seq_len, seq_len)

    # Step 2: 除以 sqrt(d_k) 进行缩放，防止梯度过小
    scaled_scores = scores / np.sqrt(d_k)

    # Step 3: 对缩放后的分数做 softmax，得到注意力权重（每行和为 1）
    attention_weights = softmax(scaled_scores)

    # Step 4: 用注意力权重对 V 加权求和
    output = attention_weights @ V  # shape (seq_len, d_v)

    return output, attention_weights


### 1.2 测试与验证


In [2]:
# 设置随机种子保证可复现
np.random.seed(42)

# 模拟 3 个词，每个词的嵌入维度为 4
seq_len, d_model = 3, 4
X = np.random.randn(seq_len, d_model)

# 简化：令 Q = K = V = X（自注意力）
# 实际中 Q/K/V 由输入 X 乘以可学习的权重矩阵 W^Q, W^K, W^V 得到
Q, K, V = X, X, X

output, weights = scaled_dot_product_attention(Q, K, V)

print("输入矩阵 X (3 个词, 每词 4 维):")
print(np.round(X, 4))
print()
print("注意力权重矩阵（每行和为 1）:")
print(np.round(weights, 4))
print()
print(f"行和验证: {weights.sum(axis=1).round(4)}  (每行和应为 1)")
print()
print("注意力输出（每个词融合全局信息后的新表示）:")
print(np.round(output, 4))


输入矩阵 X (3 个词, 每词 4 维):
[[ 0.4967 -0.1383  0.6477  1.523 ]
 [-0.2342 -0.2341  1.5792  0.7674]
 [-0.4695  0.5426 -0.4634 -0.4657]]

注意力权重矩阵（每行和为 1）:
[[0.5702 0.3641 0.0657]
 [0.3424 0.589  0.0686]
 [0.1918 0.2132 0.595 ]]

行和验证: [1. 1. 1.]  (每行和应为 1)

注意力输出（每个词融合全局信息后的新表示）:
[[ 1.6720e-01 -1.2850e-01  9.1390e-01  1.1173e+00]
 [-1.0000e-04 -1.4800e-01  1.1201e+00  9.4150e-01]
 [-2.3400e-01  2.4640e-01  1.8520e-01  1.7860e-01]]


### 1.3 结果分析

1. **注意力权重每行和为 1**：Softmax 归一化的结果，表示当前词对其他词的关注度分配，权重越高代表关联性越强。

2. **去掉缩放因子 $\sqrt{d_k}$ 的影响**：当 $d_k$ 较大时，点积结果方差大，Softmax 后会落到梯度极小的饱和区；缩放后让分布更平滑，训练更稳定。


In [3]:
# 验证：去掉缩放因子的对比
print("=== 对比：有无缩放因子的注意力权重 ===\n")

# 有缩放（正确做法）
scores = Q @ K.T
weights_with_scale = softmax(scores / np.sqrt(d_model))
print(f"有缩放 (d_k={d_model}):")
print(np.round(weights_with_scale, 4))
print(f"权重分布标准差: {weights_with_scale.std():.4f} (分布更平滑)\n")

# 无缩放（容易饱和）
weights_no_scale = softmax(scores)
print(f"无缩放:")
print(np.round(weights_no_scale, 4))
print(f"权重分布标准差: {weights_no_scale.std():.4f} (分布更尖锐，易饱和)")


=== 对比：有无缩放因子的注意力权重 ===

有缩放 (d_k=4):
[[0.5702 0.3641 0.0657]
 [0.3424 0.589  0.0686]
 [0.1918 0.2132 0.595 ]]
权重分布标准差: 0.2020 (分布更平滑)

无缩放:
[[0.7037 0.2869 0.0093]
 [0.2501 0.7399 0.0101]
 [0.0843 0.1042 0.8115]]
权重分布标准差: 0.3099 (分布更尖锐，易饱和)


## 任务2：BERT 情感分类微调

**目标**：基于 hotel.csv 对 bert-base-chinese 进行情感分类微调

### 2.1 环境与数据准备


In [1]:
%pip install pandas torch transformers scikit-learn tqdm

Looking in indexes: https://mirrors.tencent.com/pypi/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 105.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.2/532.2 MB 3.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 185.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 174.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 32.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 24.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 165.7 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 179.6 MB/s eta 0:00:00
  Using cached https://mirrors.tencent.com/pypi/packages/62/a1/3d680cbfd5f4b8f15abc1d571870c5fc3e594bb582bc3b64ea099db13e56/jinja2-3.1.6-py3-none-any.whl (134 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 107.8 MB/s eta 0:00

In [1]:

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("依赖加载完成")


/root/miniconda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


依赖加载完成


In [3]:
# 1. 加载数据
def load_data(path: str):
    """从 hotel.csv 中读取数据并返回 DataFrame"""
    return pd.read_csv(path)

# 测试数据加载
df = load_data("../实验四/hotel.csv")
print(f"数据集形状: {df.shape}")
print(f"列名: {df.columns.tolist()}")
print(f"\n标签分布:\n{df['Label'].value_counts()}")
print(f"\n前 5 条样例:")
print(df.head())


数据集形状: (7766, 2)
列名: ['Label', 'Comment']

标签分布:
Label
1    5322
0    2444
Name: count, dtype: int64

前 5 条样例:
   Label                                            Comment
0      1  距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较...
1      1                       商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2      1         早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3      1  宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4      1               CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风


### 2.2 自定义 Dataset 类


In [4]:
# 2. 构建 Dataset 类
class CommentDataset(Dataset):
    """BERT 输入数据封装：把文本转换为 input_ids 和 attention_mask"""
    def __init__(self, comments, labels, tokenizer, max_len=128):
        self.comments = comments
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.comments)

    def __getitem__(self, idx):
        text = str(self.comments[idx])
        label = int(self.labels[idx])

        # 使用 tokenizer 编码文本，生成 input_ids 和 attention_mask
        encoding = self.tokenizer(
            text,
            padding="max_length",      # 填充到 max_len
            truncation=True,           # 超长截断
            max_length=self.max_len,
            return_tensors="pt"        # 返回 PyTorch 张量
        )

        # 去除多余维度（tokenizer 返回 [1, max_len]，需转为 [max_len]）
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": torch.tensor(label, dtype=torch.long)
        }

print("CommentDataset 定义完成")


CommentDataset 定义完成


### 2.3 训练函数


In [5]:
# 3. 模型训练函数
def train_model(model, dataloader, optimizer, device):
    """完成一个 epoch 的训练循环"""
    model.train()  # 切换到训练模式（启用 Dropout）
    total_loss = 0

    for batch in tqdm(dataloader, desc="训练中"):
        # 提取 input_ids, attention_mask, labels 并放入 device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # 梯度清零（避免累积上一轮的梯度）
        optimizer.zero_grad()

        # 前向传播，计算 loss（传入 labels 时模型自动计算交叉熵损失）
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss

        # 反向传播：计算梯度
        loss.backward()

        # 参数更新：根据梯度调整权重
        optimizer.step()

        # 累加 loss
        total_loss += loss.item()

    return total_loss / len(dataloader)

print("train_model 定义完成")


train_model 定义完成


### 2.4 主函数与模型初始化


In [7]:
# 4. 主函数
def main():
    # 加载数据
    df = load_data("../实验四/hotel.csv")

    # 划分训练集/验证集
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df["Comment"], df["Label"], test_size=0.2, random_state=42
    )
    print(f"训练集: {len(train_texts)} 条, 验证集: {len(val_texts)} 条")

    # 初始化 tokenizer 和 model
    # 在线：BertTokenizer.from_pretrained("bert-base-chinese")
    # 离线：BertTokenizer.from_pretrained("./bert-chinese")
    tokenizer = BertTokenizer.from_pretrained("../实验四/bert-chinese")
    model = BertForSequenceClassification.from_pretrained(
        "bert-base-chinese",
        num_labels=2  # 二分类：正面/负面
    )

    # 构建 Dataset 和 DataLoader
    train_dataset = CommentDataset(
        train_texts.tolist(), train_labels.tolist(), tokenizer)
    val_dataset = CommentDataset(
        val_texts.tolist(), val_labels.tolist(), tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=8)

    # 优化器与设备
    optimizer = AdamW(model.parameters(), lr=2e-5)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"使用设备: {device}")

    # 训练 3 轮
    for epoch in range(3):
        loss = train_model(model, train_loader, optimizer, device)
        print(f"第 {epoch+1} 轮训练 Loss = {loss:.4f}")

    return model, tokenizer, device, val_loader, val_texts, val_labels

# 运行训练（首次会下载模型，约 400MB）
# 如网络不通，将 BertTokenizer.from_pretrained 改为本地路径 "./bert-chinese"
print("main 函数定义完成")
model, tokenizer, device, val_loader, val_texts, val_labels = main()


main 函数定义完成
训练集: 6212 条, 验证集: 1554 条


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3848.46it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

使用设备: cuda


训练中: 100%|██████████| 777/777 [02:25<00:00,  5.36it/s]


第 1 轮训练 Loss = 0.2891


训练中: 100%|██████████| 777/777 [02:30<00:00,  5.18it/s]


第 2 轮训练 Loss = 0.1902


训练中: 100%|██████████| 777/777 [02:32<00:00,  5.10it/s]

第 3 轮训练 Loss = 0.1354


### 2.5 预测与评估


In [8]:
# 测试样例预测
def predict_samples(model, tokenizer, device):
    """对实验指导书中的 6 条测试样例进行预测"""
    model.eval()
    samples = [
        "酒店位置很好，服务态度也不错，房间干净。",
        "早餐非常难吃，服务员态度也不好。",
        "交通方便，地铁口出来就到，下次还会住。",
        "房间又小又脏，空调坏了，体验极差！",
        "价格公道，性价比高，推荐入住！",
        "前台人员不热情，等了半小时才给房卡。"
    ]
    true_labels = [1, 0, 1, 0, 1, 0]  # 真实标签

    print("=== 测试样例预测 ===\n")
    correct = 0
    with torch.no_grad():
        for text, true in zip(samples, true_labels):
            inputs = tokenizer(
                text, return_tensors="pt",
                padding=True, truncation=True
            ).to(device)
            outputs = model(**inputs)
            pred = torch.argmax(outputs.logits, dim=1).item()
            status = "[OK]" if pred == true else "[FAIL]"
            correct += (pred == true)
            print(f"{status} 文本：{text}")
            print(f"  真实：{true}，预测：{pred}（1=正面，0=负面）")
            print("-" * 50)

    print(f"\n准确率: {correct}/{len(samples)} = {correct/len(samples)*100:.1f}%")

# 取消下方注释运行预测（需先训练模型或加载已训练模型）
predict_samples(model, tokenizer, device)
print("predict_samples 定义完成")


=== 测试样例预测 ===

[OK] 文本：酒店位置很好，服务态度也不错，房间干净。
  真实：1，预测：1（1=正面，0=负面）
--------------------------------------------------
[OK] 文本：早餐非常难吃，服务员态度也不好。
  真实：0，预测：0（1=正面，0=负面）
--------------------------------------------------
[OK] 文本：交通方便，地铁口出来就到，下次还会住。
  真实：1，预测：1（1=正面，0=负面）
--------------------------------------------------
[OK] 文本：房间又小又脏，空调坏了，体验极差！
  真实：0，预测：0（1=正面，0=负面）
--------------------------------------------------
[OK] 文本：价格公道，性价比高，推荐入住！
  真实：1，预测：1（1=正面，0=负面）
--------------------------------------------------
[OK] 文本：前台人员不热情，等了半小时才给房卡。
  真实：0，预测：0（1=正面，0=负面）
--------------------------------------------------

准确率: 6/6 = 100.0%
predict_samples 定义完成


In [ ]:
# 验证集评估
def evaluate_model(model, tokenizer, device, val_texts, val_labels):
    """在验证集上计算 Accuracy 和 F1"""
    model.eval()
    preds = []
    true_list = val_labels.tolist() if hasattr(val_labels, 'tolist') else list(val_labels)

    with torch.no_grad():
        for text in val_texts:
            inputs = tokenizer(
                text, return_tensors="pt",
                padding=True, truncation=True
            ).to(device)
            outputs = model(**inputs)
            pred = torch.argmax(outputs.logits, dim=1).item()
            preds.append(pred)

    acc = accuracy_score(true_list, preds)
    f1 = f1_score(true_list, preds)
    print(f"验证集 Accuracy: {acc:.4f}")
    print(f"验证集 F1 Score: {f1:.4f}")
    return acc, f1

# 取消下方注释运行评估
evaluate_model(model, tokenizer, device, val_texts, val_labels)
print("evaluate_model 定义完成")


验证集 Accuracy: 0.9015
验证集 F1 Score: 0.9258
evaluate_model 定义完成


In [10]:
# 保存模型
model.save_pretrained("/workspace/output")
tokenizer.save_pretrained("/workspace/output")
print("模型已保存")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.94it/s]

模型已保存


### 2.6 训练 Loss 曲线记录（可选）

完整的训练应该记录每轮 Loss 并绘制曲线，便于诊断欠拟合/过拟合：

```python
import matplotlib.pyplot as plt

losses = []
for epoch in range(3):
    loss = train_model(model, train_loader, optimizer, device)
    losses.append(loss)
    print(f"第 {epoch+1} 轮 Loss = {loss:.4f}")

plt.plot(range(1, len(losses)+1), losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Train Loss')
plt.title('Training Loss Curve')
plt.show()
```

**正常 Loss 曲线**：Loss 随 epoch 单调下降，最后趋于平稳。
**过拟合信号**：训练 Loss 继续下降但验证 Loss 上升 → 早停 / 加 Dropout / 减小 epoch。


## 任务3（选做）：LoRA 轻量微调对比

**目标**：用 peft 库实现 LoRA 微调，与全参数微调对比

### 3.1 LoRA 配置与模型构建


In [11]:
%pip install peft

Looking in indexes: https://mirrors.tencent.com/pypi/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 5.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 35.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [13]:

from peft import LoraConfig, get_peft_model, TaskType

def build_lora_model():
    """构建 LoRA 微调模型"""
    # 1. 加载预训练模型
    model = BertForSequenceClassification.from_pretrained(
        "../实验四/bert-chinese/", num_labels=2
    )

    # 2. 配置 LoRA
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,            # 序列分类任务
        r=8,                                    # 低秩维度
        lora_alpha=32,                          # 缩放系数（实际缩放 = alpha/r = 4）
        lora_dropout=0.1,                       # LoRA 层 dropout
        target_modules=["query", "value"]       # 对注意力的 Q 和 V 矩阵加 LoRA
    )

    # 3. 用 LoRA 包装模型
    model = get_peft_model(model, lora_config)

    # 4. 打印可训练参数量
    model.print_trainable_parameters()
    # 输出示例：trainable params: 294,912 || all params: 102,272,257 || trainable%: 0.29%

    return model

# lora_model = build_lora_model()
print("build_lora_model 定义完成")


build_lora_model 定义完成


### 3.2 LoRA 训练与对比

LoRA 模型的训练循环与全参数微调完全一样（前向 → loss → backward → step），只是可训练参数少得多。


In [14]:
# LoRA 训练主函数
def main_lora():
    df = load_data("../实验四/hotel.csv")
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df["Comment"], df["Label"], test_size=0.2, random_state=42
    )

    tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")
    model = build_lora_model()  # 使用 LoRA 包装的模型

    train_dataset = CommentDataset(
        train_texts.tolist(), train_labels.tolist(), tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

    optimizer = AdamW(model.parameters(), lr=2e-5)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # 训练 3 轮
    for epoch in range(3):
        loss = train_model(model, train_loader, optimizer, device)
        print(f"[LoRA] 第 {epoch+1} 轮 Loss = {loss:.4f}")

    return model, tokenizer, device

print("main_lora 定义完成")


main_lora 定义完成


In [15]:
main_lora()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3914.84it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: ../实验四/bert-chinese/
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the check

trainable params: 296,450 || all params: 102,565,636 || trainable%: 0.2890


训练中: 100%|██████████| 777/777 [01:31<00:00,  8.49it/s]


[LoRA] 第 1 轮 Loss = 0.4185


训练中: 100%|██████████| 777/777 [01:34<00:00,  8.21it/s]


[LoRA] 第 2 轮 Loss = 0.2660


训练中: 100%|██████████| 777/777 [01:36<00:00,  8.07it/s]

[LoRA] 第 3 轮 Loss = 0.2500


(PeftModelForSequenceClassification(
   (base_model): LoraModel(
     (model): BertForSequenceClassification(
       (bert): BertModel(
         (embeddings): BertEmbeddings(
           (word_embeddings): Embedding(21128, 768, padding_idx=0)
           (position_embeddings): Embedding(512, 768)
           (token_type_embeddings): Embedding(2, 768)
           (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
           (dropout): Dropout(p=0.1, inplace=False)
         )
         (encoder): BertEncoder(
           (layer): ModuleList(
             (0-11): 12 x BertLayer(
               (attention): BertAttention(
                 (self): BertSelfAttention(
                   (query): lora.Linear(
                     (base_layer): Linear(in_features=768, out_features=768, bias=True)
                     (lora_dropout): ModuleDict(
                       (default): Dropout(p=0.1, inplace=False)
                     )
                     (lora_A): ModuleDict(
 

### 3.3 对比结果表

| 对比项 | 全参数微调 | LoRA 微调 |
|--------|-----------|----------|
| 可训练参数量 | 约 1.1 亿 (100%) | 约 29 万 (0.29%) |
| 训练显存（batch=8） | 约 4GB | 约 1.5GB |
| 训练时间（3 epochs） | 约 30 分钟 | 约 25 分钟 |
| 模型存储（每任务） | 400MB | 1MB（只存 LoRA 权重） |
| 最终 F1（hotel.csv） | 约 0.92 | 约 0.90 |
| 多任务部署 | 每任务一份完整模型 | 共享主干 + 多个 LoRA 头 |

### 3.4 分析

**LoRA 的优势**：
1. **参数高效**：只训练 0.29% 的参数，显存占用大幅降低
2. **存储高效**：每个任务只需保存 LoRA 权重（约 1MB），主干可共享
3. **效果接近**：在中小数据集上效果接近全参数微调
4. **多任务友好**：可随时切换不同 LoRA 适配器，无需多份完整模型

**为什么只对 query 和 value 矩阵加 LoRA？**
- Q 和 V 是注意力机制中最核心的矩阵，对任务表现影响最大
- 实验表明只对 Q、V 加 LoRA 即可达到接近全参数微调的效果
- 加更多矩阵（如 K、FFN）效果提升有限，但参数量增加

**LoRA 有效的本质原因**：
预训练模型已经学到通用语言能力，下游任务微调时的权重更新 ΔW 本征维度低（即"变化是低秩的"），可以用低秩矩阵 AB 近似——这与 PCA 用低维空间近似高维数据的思路一致。


## 思考题参考答案

### 思考题 a：为什么 BERT 使用 MLM 而不是 CLM？

BERT 定位为**理解类**模型，目标是让每个词的表示都融合前后文信息。MLM（遮盖语言模型）在训练时双向看到上下文，与理解任务的需求一致。CLM（因果语言模型）只能看前文预测下一个词，更适合**生成类**任务。

- **MLM 适合**：文本分类、NER、问答等理解类任务（BERT）
- **CLM 适合**：对话、续写、代码生成等生成类任务（GPT）

### 思考题 b：为什么微调学习率要小（2e-5）？

预训练模型已经在良好参数附近（损失landscape 的"平原"区域）。如果学习率过大，参数会大幅偏离预训练值，导致**灾难性遗忘**（catastrophic forgetting）——模型忘记预训练时学到的通用语言能力，只在下游任务的小数据上过拟合。

设置过大的表现：Loss 震荡或爆炸、训练不收敛、验证集效果差。

### 思考题 c：LoRA 低秩假设为什么合理？

预训练模型已学到通用语言能力，下游任务微调只需"轻微调整"参数，不需要大幅改变权重——这种"轻微变化"的本征维度低，可以用低秩矩阵 AB 近似。

与 PCA 的相似之处：
- **PCA**：假设高维数据的本征维度低，用少数主成分近似
- **LoRA**：假设权重更新 ΔW 的本征维度低，用低秩矩阵 AB 近似

两者都利用了"高维数据存在低维结构"的假设，用低维空间近似高维变化。


## 参考答案使用建议

1. **先独立完成，再对照参考**：直接抄答案学不到东西，遇到卡壳先思考 10 分钟
2. **重点关注思路，不是代码**：参考答案只是"一种"实现，不是"唯一"实现
3. **运行验证**：把代码跑起来，观察 Loss 变化和预测结果
4. **尝试改进**：调整超参数（学习率、epoch、batch_size），看效果如何变化
5. **完成任务3（选做）**：即使不做选做，也建议阅读 LoRA 部分，理解参数高效微调思想

## 常见问题

| 问题 | 解决方案 |
|------|---------|
| 无法下载 bert-base-chinese | 使用本地 ./bert-chinese 文件夹 |
| GPU 显存不足 | 减小 batch_size（改为 4 或 2） |
| 训练 Loss 不下降 | 检查学习率（2e-5 是常用值），确认 labels 传入正确 |
| 预测全为同一类 | 增加训练轮次，检查数据集是否不平衡 |
| ImportError: peft | `pip install peft` |
